# HCL Technologies India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** careers.hcltech.com

**ATS Detection:** Automatic API + Selenium fallback

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml playwright -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path

# Add scripts dir to path so we can import scraper_utils
SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from scraper_utils import *
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

# ── LOCATION CONFIG ──────────────────────────────────────────────────────────
# Change to "" to scrape globally (all countries).
# The matching pipeline's pre-filter handles India-specific narrowing.
# Set to "India" here only if you want to reduce volume at scrape time.
LOCATION_FILTER = ""
COUNTRY_CODE   = ""  # e.g. "in" for SmartRecruiters country= param; "" = all
# ─────────────────────────────────────────────────────────────────────────────

print("Imports loaded. Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Location filter: '{LOCATION_FILTER}' (empty = broad/global scraping)")


scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-03-31 23:37:03
Location filter: '' (empty = broad/global scraping)


In [3]:
COMPANY = "HCL_Technologies"
OUTPUT_DIR = get_output_dir(COMPANY)
print(f"Output directory: {OUTPUT_DIR}")


Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/HCL_Technologies/Outputs/2026_03_31


In [4]:
print("=" * 60)
print("HCL TECHNOLOGIES INDIA JOB SCRAPER")
print("Primary: Workday API + Selenium fallback")
print("=" * 60)

from selenium.webdriver.common.by import By

hcl_jobs = []

# Try Workday first
try:
    hcl_jobs = scrape_workday(
        tenant="hcltech",
        instance="wd3",
        career_site="HCLTech",
        company_name="HCL Technologies",
        industry="IT Services",
        location_filter=LOCATION_FILTER,
        max_jobs=500
    )
except Exception as e:
    print(f"  Workday approach failed: {e}")

# Fallback: Selenium on careers.hcltech.com
if len(hcl_jobs) < 5:
    print("\n  Trying Selenium on careers.hcltech.com...")
    driver = setup_selenium()
    try:
        driver.get("https://careers.hcltech.com/jobs?location=India")
        time.sleep(8)
        soup = BeautifulSoup(driver.page_source, "lxml")

        cards = soup.select("[class*='job-card'], [class*='job-listing'], [class*='career-card'], .views-row, a[href*='/job']")
        for card in cards:
            title_el = card.select_one("h2, h3, h4, a, [class*='title']")
            title = title_el.get_text(strip=True) if title_el else ""
            loc_el = card.select_one("[class*='location'], [class*='city']")
            loc = loc_el.get_text(strip=True) if loc_el else "India"
            link = card.select_one("a[href]")
            href = link.get("href", "") if link else ""

            if title:
                hcl_jobs.append({
                    "job_id": href.split("/")[-1] if href else str(len(hcl_jobs)),
                    "title": title,
                    "company_name": "HCL Technologies",
                    "raw_jd_text": card.get_text(" ", strip=True),
                    "location_city": loc.split(",")[0].strip(),
                    "industry": "IT Services",
                    "date_posted": datetime.now().strftime("%Y-%m-%d"),
                    "is_active": True,
                    "job_url": href if href.startswith("http") else f"https://careers.hcltech.com{href}" if href else "",
                    "business_unit": "",
                    "source_platform": "HCL Selenium fallback",
                })
    except Exception as e:
        print(f"  Selenium failed: {e}")
    finally:
        driver.quit()

print(f"Total HCL India jobs: {len(hcl_jobs)}")


HCL TECHNOLOGIES INDIA JOB SCRAPER
Primary: Workday API + Selenium fallback
  Scraping HCL Technologies via Workday API: https://hcltech.wd3.myworkdayjobs.com/wday/cxs/hcltech/HCLTech/jobs
  Mode: BROAD (no location filter — fetching all global jobs)


  [ERROR] HTTP 422 at offset 0
  Total HCL Technologies India jobs: 0

  Trying Selenium on careers.hcltech.com...


Total HCL India jobs: 0


In [5]:
df_hcl = save_results(hcl_jobs, "HCL Technologies", OUTPUT_DIR)
if df_hcl is not None:
    print(f"\nSample jobs:")
    cols = ["title","location_city","seniority_level","business_unit","job_url"]
    cols = [c for c in cols if c in df_hcl.columns]
    print(df_hcl[cols].head(10).to_string())


  [WARN] No jobs found for HCL Technologies
